### Initialize instrumentation

In [ ]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor, LeCroyScope
from Equipments.MCBOX import generate_test_voltages
from DataProcessing.Plot import PlotSaver
import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

In [ ]:
# set up readout and plotting
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [ ]:
# open list of available instruments
rm = pyvisa.ResourceManager()
print(rm.list_resources())

In [ ]:
# connect to BNC 575 pulse generator (for controlling experimental time sequence)
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
# connect to WeederTech WTMCD-M driver (for 480 stepper motor)
weeder = Weeder.Weeder("ASRL1::INSTR")

In [ ]:
# connect to MC USB-3114 (for compensation voltages)
mcbox = MCBOX.MCBOX(find_device = "USB-3114")

### Set compensation voltages and define timing sequence, set up photon counter

In [ ]:
# vset = np.array([0.04, -0.025, 0.222])
vset = np.array([0.4, -0.05, -0.015])
mcbox.set_voltage_Nchan(channels = [0,1,6,7,10,11], 
                        voltages = [0,-vset[2],vset[1],-vset[1],vset[0],-vset[0]], bipolar = True)

In [ ]:
# define timing sequence

############################## current pulsed sequence ##############################
# pulse_arrangement = [["A", 1300e-6, 0, 2, 1],                 # MOT AOM
#                      [["B", 15e-6, 40e-6, 2, 1],              # 780 AOM
#                      ["C", 15e-6, 40e-6, 6, -1]],             # 480 switch
#                      [["D", 26e-6, 0, 6, 1],                  # AWG
#                      ["E", 26e-6, 0, 6, 1]],                  # SR400 gated counter
#                     ]

############################## current pulsed sequence (no counter) ################
pulse_arrangement = [["A", 1300e-6, 0, 2, 1],                 # MOT AOM
                     [["B", 15e-6, 40e-6, 2, 1],              # 780 AOM
                     ["C", 15e-6, 40e-6, 6, -1]],             # 480 switch
                     ["D", 26e-6, 0, 6, 1],                   # AWG
                    ]

# pulse_arrangement = [
#                     [["D", 26e-06, 1355e-06, 6, 1],             # AWG; ampl changed from 6 V
#                     ["E", 26e-06, 1355e-06, 6, 1]]              # gated photon counter
#                     ]
T = 1600e-6  # sequence period; changed from 1500 us

# cycle_number = 30

# photon counter gate width and delay
# gate_width = 130e-6 # changed from 20 us
# gate_delay = 3e-6  # unchanged

# notes = {"gate_width": gate_width, "gate_delay": gate_delay}
notes = {}

In [ ]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

In [ ]:
bnc575.start_pulses()

### Find the Rydberg resonance

In [ ]:
# find current stepper motor position
weeder.position(header = "A", query = True)

In [ ]:
# begin stepper motor at initial position 6000, then move it out to increase the laser frequency
weeder.move("A", position = 6000, progress = True)

In [ ]:
# used to move motor incrementally to zero in on the location of a resonance
weeder.advance(header = "A", num_step = 10)

### Collect oscilloscope traces

In [ ]:
# connect to LeCroy 9354AM oscilloscope (for displaying AWG pulse and Rydberg counts for both states)
lecroy = LeCroyScope.LeCroyScope('GPIB0::3::INSTR', timeout = 20000)
lecroy.setup_transfer()

In [ ]:
# general signal collection parameters
N = 500           # number of scope sweeps averaged
max_points = 2500 # maximum number of math channel points
V_mcp = -700.0    # MCP front voltage [V]
V_tip = 43.0      # max SSFI voltage [V]; CHANGE THIS!!
dist = 2.7        # approximate distance between TIP and MOT [mm]

In [ ]:
# use for zooming into plot
def closest_idx(arr, value):
    idx = np.searchsorted(arr, value)
    idx = np.clip(idx, 1, len(arr) - 1)
    # Pick the closer of the two neighbors
    if value - arr[idx - 1] < arr[idx] - value:
        return idx - 1
    return idx

#### Collect background trace

In [ ]:
# collect averaged scope trace
tbg, vbg, ibg = lecroy.read_waveform("TA")         # averaged background
tfi, vfi, ifi = lecroy.read_waveform("C3")         # AWG/SSFI voltage

In [ ]:
# plot averaged scope trace
fig1, ax1 = plt.subplots()

# AWG/SSFI pulse
color = 'tab:black'
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("AWG/SSFI amplitude (V)", color = color)
ax1.plot(tfi, vfi, color = color)
ax1.tick_params(axis = 'y', labelcolor = color)
ax1.set_title(rf'Background MCP signal and SSFI pulse')

# background spectrum
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel("Background MCP signal (V)", color = color)
ax2.plot(tbg, vbg, color = color)
ax2.tick_params(axis = 'y', labelcolor = color)

fig1.tight_layout()
# plt.grid(True)
plt.show()

In [ ]:
notes = {"Number of sweeps averaged": N, "Max averaged points": max_points, "Maximum TIP voltage (V)": V_tip,
        "MCP front voltage (V)": V_mcp, "Approximate MOT-TIP distance (mm)": dist, "Ortec connected?:": "True"}

In [ ]:
# save background trace
ps.Plot2D(x = tbg, y = None, Z = vbg,
          xlabel = "Time (s)", zlabel = "Background MCP signal (V)", notes = notes)

In [ ]:
# save AWG/SSFI pulse
ps.Plot2D(x = tfi, y = None, Z = vfi,
          xlabel = "Time (s)", zlabel = "AWG/SSFI amplitude (V)", notes = notes)

#### Collect 69 S trace

In [ ]:
state = "69 S 1/2"

In [ ]:
# collect averaged scope trace
t69, v69, i69 = lecroy.read_waveform("TA")         # averaged 69 S signal
tfi, vfi, ifi = lecroy.read_waveform("C3")         # AWG/SSFI voltage

In [ ]:
# plot averaged scope trace
fig2, ax1 = plt.subplots()

# AWG/SSFI pulse
color = 'tab:black'
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("AWG/SSFI amplitude (V)", color = color)
ax1.plot(tfi, vfi, color = color)
ax1.tick_params(axis = 'y', labelcolor = color)
ax1.set_title(rf'Rydberg spectrum for {state} (averaged over {N} sweeps)')

# 69 S 1/2 Rydberg spectrum
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel("69 S 1/2 MCP signal (V)", color = color)
ax2.plot(t69, v69, color = color)
ax2.tick_params(axis = 'y', labelcolor = color)

fig2.tight_layout()
# plt.grid(True)
plt.show()

In [ ]:
notes = {"Rydberg state": state, "Number of sweeps averaged": N, "Max averaged points": max_points, "Maximum TIP voltage (V)": V_tip,
        "MCP front voltage (V)": V_mcp, "Approximate MOT-TIP distance (mm)": dist, "Ortec connected?:": "True"}

In [ ]:
# save 69 S trace
ps.Plot2D(x = t69, y = None, Z = v69,
          xlabel = "Time (s)", zlabel = "MCP 69 S signal (V)", notes = notes)

#### Collect 70 S trace

In [ ]:
state = "70 S 1/2"

In [ ]:
# collect averaged scope trace
t70, v70, i70 = lecroy.read_waveform("TA")         # averaged 70 S signal
tfi, vfi, ifi = lecroy.read_waveform("C3")         # AWG/SSFI voltage

In [ ]:
# plotting
fig3, ax1 = plt.subplots()

# AWG/SSFI pulse
color = 'tab:black'
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("AWG/SSFI amplitude (V)", color = color)
ax1.plot(tfi, vfi, color = color)
ax1.tick_params(axis = 'y', labelcolor = color)
ax1.set_title(rf'Rydberg spectrum for {state} (averaged over {N} sweeps)')

# 70 S 1/2 Rydberg spectrum
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel("70 S 1/2 MCP signal (V)", color = color)
ax2.plot(t70, v70, color = color)
ax2.tick_params(axis = 'y', labelcolor = color)

fig3.tight_layout()
# plt.grid(True)
plt.show()

In [ ]:
notes = {"Rydberg state": state, "Number of sweeps averaged": N, "Max averaged points": max_points, "Maximum TIP voltage (V)": V_tip,
        "MCP front voltage (V)": V_mcp, "Approximate MOT-TIP distance (mm)": dist, "Ortec connected?:": "True"}

In [ ]:
# save 70 S trace
ps.Plot2D(x = t70, y = None, Z = v70,
          xlabel = "Time (s)", zlabel = "MCP 70 S signal (V)", notes = notes)

### Shut down instrumentation

In [ ]:
# reset voltages to zero
vset = np.array([0.0,0.0,0.0])
mcbox.set_voltage_Nchan(channels = [0,1,6,7,10,11], 
                        voltages = [0,-vset[2],vset[1],-vset[1],vset[0],-vset[0]], bipolar = True)

In [ ]:
# save current stepper motor position
weeder.save(header = "A", query = True)

In [ ]:
# OPTIONAL: disables pulses from BNC box 
# (can also be done by pressing RUN/STOP button or individual channel buttons on device)
bnc575.disable_all()

In [ ]:
# OPTIONAL: disarms the pulse generator 
# (can also be done by pressing RUN/STOP button on device)
bnc575.disarm_all()

In [ ]:
# disconnect from pulse generator
bnc575.pyvisa.close()

In [ ]:
# disconnect from SRS photon counter
sr400.pyvisa.close()

In [ ]:
# disconnect from oscillscope
lecroy.close()